In [0]:
spark

##### TURN OFF AQE

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "false")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

#### Data Reading

In [0]:
df = spark.read.format('csv')\
  .option('inferSchema', 'true')\
  .option('header', 'true')\
  .load('/FileStore/rawdata/BigMart_Sales.csv')
display(df)

#### Get No of Partitions

In [0]:
df.rdd.getNumPartitions()
#output : 1

#### Changing DEFAULT Partition Size to 128 kb

In [0]:
spark.conf.set("spark.sql.files.maxPartitionBytes",131072)

In [0]:
df.rdd.getNumPartitions()
#output : 7

In [0]:
from pyspark.sql.functions import *

In [0]:
df.withColumn("partition_ID",spark_partition_id()).display()

#### Data Writing

In [0]:
df.write.format("parquet")\
    .mode("append")\
    .option("path","/FileStore/rawdata/parquetWrite")\
    .save()


In [0]:
df_new = spark.read.format("parquet")\
            .load("/FileStore/rawdata/parquetWrite")
df_new.display()

In [0]:
df_new =df_new.filter(col("Outlet_Location_Type")=='Tier 1')
df_new.display()

In [0]:
df.write.format("parquet")\
    .mode("append")\
    .partitionBy("Outlet_Location_Type")\
    .option("path","/FileStore/rawdata/parquetWriteOpt")\
    .save()

In [0]:
df_opt=spark.read.format("parquet")\
            .load("/FileStore/rawdata/parquetWriteOpt")

df_opt = df_opt.filter(col("Outlet_Location_Type")=='Tier 1')
df_opt.display()


### JOINS Optimization 

#### Broadcast Join